In [ ]:
from pyspark.sql import functions as F

from nyc_taxi_trips_etl.utils.pipeline_params_parsing import (
    parse_and_expand_year_months,
)

In [ ]:
# pipeline / run parameters
year_months_to_extract = dbutils.widgets.get("year_months_to_extract")

catalog = dbutils.widgets.get("catalog")
run_id = dbutils.widgets.get("run_id")
data_collection_started_at = dbutils.widgets.get("data_collection_started_at")

In [ ]:
year_months = parse_and_expand_year_months(year_months_to_extract)

In [ ]:
trip_data_df = spark.read.parquet(
    f"/Volumes/{catalog}/landing/yellow_trip_data/{run_id}/"
)

trip_data_df_with_metadata = trip_data_df.withColumns(
    {
        "source_file_path": F.col("_metadata.file_path"),
        "ingestion_timestamp": F.to_timestamp(F.lit(data_collection_started_at)),
        "pipeline_run_id": F.lit(run_id),
        "year_month": F.regexp_extract(
            F.col("_metadata.file_name"), r"yellow_tripdata_(\d{4}-\d{2})", 1
        ),
    }
)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,source_file_path,ingestion_timestamp,year_month
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1,1.60,1,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0,dbfs:/Volumes/nyctaxi_dev/landing/yellow_trip_data/yellow_tripdata_2025-01.parquet,2026-06-26 12:06:20.670776,2025-01
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1,0.50,1,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0,dbfs:/Volumes/nyctaxi_dev/landing/yellow_trip_data/yellow_tripdata_2025-01.parquet,2026-06-26 12:06:20.670776,2025-01
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1,0.60,1,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0,dbfs:/Volumes/nyctaxi_dev/landing/yellow_trip_data/yellow_tripdata_2025-01.parquet,2026-06-26 12:06:20.670776,2025-01
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3,0.52,1,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0,dbfs:/Volumes/nyctaxi_dev/landing/yellow_trip_data/yellow_tripdata_2025-01.parquet,2026-06-26 12:06:20.670776,2025-01
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3,0.66,1,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0,dbfs:/Volumes/nyctaxi_dev/landing/yellow_trip_data/yellow_tripdata_2025-01.parquet,2026-06-26 12:06:20.670776,2025-01


In [ ]:
(
    trip_data_df_with_metadata.write.mode("overwrite")
    .option(
        "replaceWhere",
        f"""year_month in ({",".join([f"'{ym}'" for ym in year_months])})""",
    )
    .saveAsTable(f"{catalog}.bronze.yellow_trip_data")
)